# 주파수 도메인 특징 추출: FFT + DWT (Colab 버전)

**입력**: `data/sequences/sequences.h5` → `X_flat` (N, 60, 20)  
**출력**: `data/sequences/freq_features.h5` → 각 split별 FFT + DWT 피처

**파이프라인**
```
X_flat[:, :, 0]  ← Adj_Close 시계열 (60일)
    ├── FFT  → 상위 주파수 성분 (진폭, 주파수, 스펙트럼 에너지)
    └── DWT  → 레벨별 추세/노이즈 성분 (근사 + 세부 계수 통계)
         → freq_features.h5 저장 (배치 처리, 메모리 절약)
         → LightGBM으로 유효성 검증
```

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pywavelets lightgbm h5py tqdm

import gc
import warnings
import numpy as np
import pandas as pd
import h5py
import pywt
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

DRIVE_ROOT = Path('/content/drive/MyDrive/grad_project')
SEQ_H5     = DRIVE_ROOT / 'data/sequences/sequences.h5'
FREQ_H5    = DRIVE_ROOT / 'data/sequences/freq_features.h5'

assert SEQ_H5.exists(), f'파일 없음: {SEQ_H5}'
print(f'✓ 입력: {SEQ_H5}  ({SEQ_H5.stat().st_size/1e9:.2f} GB)')

# 설정
TOP_K     = 5      # FFT 상위 K개 주파수 성분
WAVELET   = 'db4'  # Daubechies-4 웨이블릿
DWT_LEVEL = 3      # 웨이블릿 분해 레벨
BATCH     = 8192   # 배치 크기 (메모리 절약)
ADJ_IDX   = 0      # X_flat에서 Adj_Close 인덱스

print(f'FFT: Top-{TOP_K} | DWT: {WAVELET} Level-{DWT_LEVEL} | Batch: {BATCH}')

## 1. 피처 함수 정의

In [ ]:
def extract_fft_features(x: np.ndarray, top_k: int = 5) -> np.ndarray:
    """
    x: (seq_len,) — 단일 시계열 (Adj_Close)
    반환: (top_k*2 + 3,) — 진폭K + 주파수K + 에너지3
    """
    n   = len(x)
    fft = np.fft.rfft(x)               # 실수 FFT
    amp = np.abs(fft)                  # 진폭 스펙트럼
    freq = np.fft.rfftfreq(n)          # 주파수 축 (0 ~ 0.5)

    # DC 성분(freq=0) 제외 후 상위 K개
    amp_no_dc = amp.copy()
    amp_no_dc[0] = 0
    top_idx = np.argsort(amp_no_dc)[::-1][:top_k]
    top_amps  = amp_no_dc[top_idx]     # 상위 K 진폭
    top_freqs = freq[top_idx]          # 상위 K 주파수

    # 스펙트럼 에너지를 저주파 / 중주파 / 고주파 3구간으로 분리
    total_energy = np.sum(amp**2) + 1e-8
    cut1 = len(amp) // 3
    cut2 = cut1 * 2
    energy_low  = np.sum(amp[:cut1]**2)       / total_energy
    energy_mid  = np.sum(amp[cut1:cut2]**2)   / total_energy
    energy_high = np.sum(amp[cut2:]**2)        / total_energy

    return np.concatenate([
        top_amps / (top_amps.sum() + 1e-8),   # 정규화된 진폭
        top_freqs,                             # 주파수 값
        [energy_low, energy_mid, energy_high], # 에너지 비율
    ]).astype(np.float32)


def extract_dwt_features(x: np.ndarray,
                          wavelet: str = 'db4',
                          level: int = 3) -> np.ndarray:
    """
    x: (seq_len,) — 단일 시계열
    반환: (level*4 + 4,) — 각 레벨 계수의 [mean, std, max, energy]
    """
    coeffs = pywt.wavedec(x, wavelet, level=level)  # [cA_n, cD_n, ..., cD_1]
    feats  = []
    for c in coeffs:
        c = np.array(c)
        energy = np.sum(c**2) / (len(c) + 1e-8)
        feats.extend([
            float(np.mean(c)),
            float(np.std(c)),
            float(np.max(np.abs(c))),
            float(energy),
        ])
    return np.array(feats, dtype=np.float32)


# 피처 차원 계산
dummy = np.random.randn(60).astype(np.float32)
FFT_DIM = len(extract_fft_features(dummy, TOP_K))
DWT_DIM = len(extract_dwt_features(dummy, WAVELET, DWT_LEVEL))
FREQ_DIM = FFT_DIM + DWT_DIM

print(f'FFT 피처 차원: {FFT_DIM}  (진폭{TOP_K} + 주파수{TOP_K} + 에너지3)')
print(f'DWT 피처 차원: {DWT_DIM}  ({DWT_LEVEL+1}레벨 × 4통계)')
print(f'총 주파수 피처: {FREQ_DIM}')

## 2. 단일 시퀀스 시각화 (직관 확인)

In [ ]:
# 샘플 1개 로드
with h5py.File(SEQ_H5, 'r') as f:
    sample_x = f['train']['X_flat'][100, :, ADJ_IDX]  # (60,)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 원본 시계열
ax = axes[0, 0]
ax.plot(sample_x, color='steelblue', lw=1.5)
ax.set_title('RevIN 정규화된 Adj_Close (60일)')
ax.set_xlabel('영업일')
ax.axhline(0, color='gray', lw=0.8, linestyle='--')

# FFT 스펙트럼
ax = axes[0, 1]
n    = len(sample_x)
fft  = np.fft.rfft(sample_x)
amp  = np.abs(fft)
freq = np.fft.rfftfreq(n)
periods = np.where(freq > 0, 1 / freq, np.inf)  # 주기(일)

amp_no_dc = amp.copy(); amp_no_dc[0] = 0
top_idx = np.argsort(amp_no_dc)[::-1][:TOP_K]

ax.bar(range(len(amp)), amp, color='steelblue', alpha=0.6, width=1)
ax.bar(top_idx, amp[top_idx], color='tomato', alpha=0.9, width=1, label=f'Top-{TOP_K}')
ax.set_title('FFT 진폭 스펙트럼')
ax.set_xlabel('주파수 인덱스')
ax.set_ylabel('진폭')
ax.legend()

# 상위 주파수 → 주기 변환
print('── 상위 주파수 성분 ──')
for i, idx in enumerate(top_idx):
    p = f'{periods[idx]:.1f}일' if periods[idx] < 999 else 'DC'
    print(f'  Top{i+1}: 주기={p:>8s}, 진폭={amp[idx]:.3f}')

# DWT 분해
coeffs = pywt.wavedec(sample_x, WAVELET, level=DWT_LEVEL)
labels = [f'cA{DWT_LEVEL}(추세)'] + [f'cD{DWT_LEVEL-i}(노이즈L{DWT_LEVEL-i})' for i in range(DWT_LEVEL)]

ax = axes[1, 0]
# cA3 재구성 (추세만)
trend = pywt.waverec([coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]], WAVELET)
trend = trend[:n]
ax.plot(sample_x, color='steelblue', lw=1.2, alpha=0.7, label='원본')
ax.plot(trend, color='red', lw=2, label=f'DWT 추세 (cA{DWT_LEVEL})')
ax.set_title('DWT 추세 성분 재구성')
ax.set_xlabel('영업일')
ax.legend()

# 레벨별 계수 에너지
ax = axes[1, 1]
energies = [np.sum(c**2) for c in coeffs]
colors   = ['tomato'] + ['steelblue'] * DWT_LEVEL
ax.bar(labels, energies, color=colors, alpha=0.8)
ax.set_title('DWT 레벨별 에너지')
ax.set_ylabel('에너지 (Σc²)')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 3. 전체 시퀀스 → 배치 처리 → freq_features.h5 저장

In [ ]:
def process_batch(X_batch: np.ndarray,
                  adj_idx: int,
                  top_k: int,
                  wavelet: str,
                  level: int) -> np.ndarray:
    """
    X_batch: (B, seq_len, n_features)
    반환:    (B, FFT_DIM + DWT_DIM)
    """
    B = X_batch.shape[0]
    out = np.empty((B, FFT_DIM + DWT_DIM), dtype=np.float32)
    for i in range(B):
        x = X_batch[i, :, adj_idx]
        fft_f = extract_fft_features(x, top_k)
        dwt_f = extract_dwt_features(x, wavelet, level)
        out[i] = np.concatenate([fft_f, dwt_f])
    return out


# 기존 파일 초기화
if FREQ_H5.exists():
    FREQ_H5.unlink()

with h5py.File(FREQ_H5, 'w') as out_f:
    with h5py.File(SEQ_H5, 'r') as in_f:
        for split in ['train', 'val', 'test']:
            N = in_f[split]['X_flat'].shape[0]
            ds = out_f.create_dataset(
                f'{split}/freq_feats',
                shape=(N, FREQ_DIM),
                dtype='float32'
            )
            print(f'[{split}] {N:,}개 시퀀스 처리 중...')
            for start in tqdm(range(0, N, BATCH), desc=split):
                end     = min(start + BATCH, N)
                X_batch = in_f[split]['X_flat'][start:end]  # (B, 60, 20)
                feats   = process_batch(X_batch, ADJ_IDX, TOP_K, WAVELET, DWT_LEVEL)
                ds[start:end] = feats
                del X_batch, feats
                gc.collect()

print(f'\n저장 완료: {FREQ_H5}  ({FREQ_H5.stat().st_size/1e6:.1f} MB)')

## 4. 피처 분포 시각화

In [ ]:
# 피처 이름 구성
fft_names = ([f'fft_amp_{i+1}'  for i in range(TOP_K)] +
             [f'fft_freq_{i+1}' for i in range(TOP_K)] +
             ['energy_low', 'energy_mid', 'energy_high'])
dwt_names = []
for lvl, label in enumerate([f'cA{DWT_LEVEL}'] + [f'cD{DWT_LEVEL-i}' for i in range(DWT_LEVEL)]):
    dwt_names += [f'dwt_{label}_mean', f'dwt_{label}_std',
                  f'dwt_{label}_max',  f'dwt_{label}_energy']
feat_names = fft_names + dwt_names

# train 1만 샘플로 분포 시각화
with h5py.File(FREQ_H5, 'r') as f:
    sample = f['train/freq_feats'][:10000]

df_feat = pd.DataFrame(sample, columns=feat_names)

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, col in zip(axes.flat, feat_names[:16]):
    vals = df_feat[col].clip(
        df_feat[col].quantile(0.01),
        df_feat[col].quantile(0.99)
    )
    ax.hist(vals, bins=50, color='steelblue', alpha=0.8, edgecolor='none')
    ax.set_title(col, fontsize=8)
    ax.tick_params(labelsize=7)
plt.suptitle('주파수 피처 분포 (train 1만 샘플, 상위 16개)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. LightGBM 유효성 검증

주파수 피처를 추가했을 때 방향성 예측 정확도가 올라가는지 확인합니다.

| 실험 | 입력 피처 |
|------|----------|
| Baseline | X_flat 평균/표준편차 (시계열 요약) |
| +FFT | Baseline + FFT 피처 |
| +DWT | Baseline + DWT 피처 |
| +FFT+DWT | Baseline + FFT + DWT 피처 |

In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, roc_auc_score

N_TRAIN_SAMPLE = 200_000  # 메모리 절약: 전체 중 일부 사용

def load_split_features(split: str, n_sample: int = None):
    """sequences.h5 + freq_features.h5 결합 로드"""
    with h5py.File(SEQ_H5, 'r') as sf, h5py.File(FREQ_H5, 'r') as ff:
        N = sf[split]['X_flat'].shape[0]
        if n_sample and n_sample < N:
            idx = np.random.choice(N, n_sample, replace=False)
            idx.sort()
        else:
            idx = np.arange(N)

        X_flat = sf[split]['X_flat'][idx]          # (N, 60, 20)
        freq   = ff[f'{split}/freq_feats'][idx]    # (N, FREQ_DIM)
        y      = sf[split]['y_dir'][idx]            # (N,)

    # 시계열 요약: 각 피처의 평균 + std (20×2 = 40차원)
    X_mean = X_flat.mean(axis=1)    # (N, 20)
    X_std  = X_flat.std(axis=1)     # (N, 20)
    base   = np.concatenate([X_mean, X_std], axis=1)   # (N, 40)

    fft_only = freq[:, :FFT_DIM]
    dwt_only = freq[:, FFT_DIM:]

    return {
        'base':        base,
        'base+fft':    np.concatenate([base, fft_only], axis=1),
        'base+dwt':    np.concatenate([base, dwt_only], axis=1),
        'base+fft+dwt':np.concatenate([base, freq],    axis=1),
    }, y


print('Train 피처 로딩 중...')
np.random.seed(42)
train_feats, y_train = load_split_features('train', N_TRAIN_SAMPLE)
val_feats,   y_val   = load_split_features('val')
print(f'Train: {y_train.shape[0]:,}개 | Val: {y_val.shape[0]:,}개')

In [ ]:
%%time

LGB_PARAMS = {
    'objective':    'binary',
    'metric':       ['binary_logloss', 'auc'],
    'n_estimators': 500,
    'learning_rate':0.05,
    'num_leaves':   63,
    'min_child_samples': 50,
    'subsample':    0.8,
    'colsample_bytree': 0.8,
    'verbose':      -1,
    'n_jobs':       -1,
    'random_state': 42,
}

results = {}
for exp_name, X_tr in train_feats.items():
    X_vl = val_feats[exp_name]
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(
        X_tr, y_train,
        eval_set=[(X_vl, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=-1)]
    )
    pred_prob = model.predict_proba(X_vl)[:, 1]
    pred_dir  = (pred_prob >= 0.5).astype(int)
    acc = accuracy_score(y_val, pred_dir)
    auc = roc_auc_score(y_val, pred_prob)
    results[exp_name] = {'acc': acc, 'auc': auc,
                         'n_features': X_tr.shape[1],
                         'best_iter': model.best_iteration_}
    print(f'[{exp_name:>15s}] Acc: {acc:.4f}  AUC: {auc:.4f}  '
          f'피처수: {X_tr.shape[1]}  iter: {model.best_iteration_}')
    del model; gc.collect()

In [ ]:
# 결과 요약 시각화
res_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'experiment'})
res_df[['acc', 'auc']] = res_df[['acc', 'auc']].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
base_acc = res_df.loc[res_df['experiment'] == 'base', 'acc'].values[0]
base_auc = res_df.loc[res_df['experiment'] == 'base', 'auc'].values[0]

for ax, metric, base_val, title in [
    (axes[0], 'acc', base_acc, 'Directional Accuracy (방향성 정확도)'),
    (axes[1], 'auc', base_auc, 'ROC-AUC'),
]:
    colors = ['tomato' if e == 'base' else 'steelblue' for e in res_df['experiment']]
    bars = ax.bar(res_df['experiment'], res_df[metric], color=colors, alpha=0.8, edgecolor='k')
    ax.axhline(base_val, color='tomato', linestyle='--', lw=1.5, label='Baseline')
    ax.set_title(title)
    ax.set_ylabel(metric.upper())
    ax.tick_params(axis='x', rotation=15)
    ax.legend(fontsize=8)
    # 차이 표시
    for bar, val in zip(bars, res_df[metric]):
        diff = val - base_val
        sign = '+' if diff >= 0 else ''
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{sign}{diff:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('주파수 피처 추가 효과 (Val set)', fontsize=13)
plt.tight_layout()
plt.show()

print(res_df.to_string(index=False))

In [ ]:
# Feature Importance (best 실험)
best_exp = res_df.loc[res_df['auc'].idxmax(), 'experiment']
print(f'최고 성능 실험: {best_exp}')

X_tr_best = train_feats[best_exp]
X_vl_best = val_feats[best_exp]

# 피처 이름
n_feat = X_tr_best.shape[1]
SEQ_FEAT_NAMES = (
    [f'mean_{i}' for i in range(20)] +
    [f'std_{i}'  for i in range(20)]
)
if 'fft' in best_exp and 'dwt' in best_exp:
    all_names = SEQ_FEAT_NAMES + feat_names
elif 'fft' in best_exp:
    all_names = SEQ_FEAT_NAMES + fft_names
elif 'dwt' in best_exp:
    all_names = SEQ_FEAT_NAMES + dwt_names
else:
    all_names = SEQ_FEAT_NAMES

best_model = lgb.LGBMClassifier(**LGB_PARAMS)
best_model.fit(
    X_tr_best, y_train,
    eval_set=[(X_vl_best, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(period=-1)]
)

imp = pd.DataFrame({
    'feature':    all_names[:n_feat],
    'importance': best_model.feature_importances_,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top30 = imp.head(30)
colors = ['tomato' if 'fft' in f or 'dwt' in f or 'energy' in f
          else 'steelblue' for f in top30['feature']]
ax.barh(top30['feature'][::-1], top30['importance'][::-1], color=colors[::-1], alpha=0.8)
ax.set_title(f'Feature Importance Top30 ({best_exp})\n빨강: 주파수 피처, 파랑: 기술적 지표 요약')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. 결과 저장 & 요약

In [ ]:
# 피처 이름 저장
pd.DataFrame({
    'index':   range(FREQ_DIM),
    'feature': feat_names,
    'group':   ['fft'] * FFT_DIM + ['dwt'] * DWT_DIM,
}).to_csv(DRIVE_ROOT / 'data/freq_feature_meta.csv', index=False)

# 실험 결과 저장
res_df.to_csv(DRIVE_ROOT / 'data/freq_validation_results.csv', index=False)

print('저장 완료:')
print(f'  freq_features.h5          → {FREQ_H5}')
print(f'  freq_feature_meta.csv     → 피처 명세')
print(f'  freq_validation_results.csv → 실험 결과')
print()
print('=' * 50)
print(' 요약')
print('=' * 50)
for _, row in res_df.iterrows():
    diff_acc = row['acc'] - base_acc
    diff_auc = row['auc'] - base_auc
    print(f"  {row['experiment']:>15s} | Acc {row['acc']:.4f} ({'+' if diff_acc>=0 else ''}{diff_acc:.4f}) "
          f"| AUC {row['auc']:.4f} ({'+' if diff_auc>=0 else ''}{diff_auc:.4f})")
print()
print('▶ 다음 단계 (7월): DTW K-Means 클러스터링 + 섹터 단위 특성 분석')